In [ ]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect('../data/gold/gold.duckdb')

# Load silver data
con.execute("""
    CREATE OR REPLACE VIEW silver AS 
    SELECT * FROM read_csv_auto('../data/silver/loans_clean.csv')
""")

print("Silver layer loaded ✓")
print(con.execute("SELECT COUNT(*) FROM silver").fetchone())

In [ ]:
# ── dim_date ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        ROW_NUMBER() OVER () AS date_key,
        issue_d::DATE AS issue_d,
        YEAR(issue_d::DATE) AS year,
        QUARTER(issue_d::DATE) AS quarter,
        MONTH(issue_d::DATE) AS month,
        STRFTIME(issue_d::DATE, '%b') AS month_name
    FROM (SELECT DISTINCT issue_d FROM silver)
    ORDER BY issue_d
""")

print("dim_date ✓")
print(con.execute("SELECT COUNT(*) FROM dim_date").fetchone())
print(con.execute("SELECT * FROM dim_date LIMIT 3").df())

## Note on dim_customer
The original plan included a `dim_customer` dimension. However, since `member_id` 
was fully null in the dataset, there is no reliable customer identifier.

Without a customer ID, grouping by customer attributes (emp_title, annual_inc, etc.) 
produces ~1.19M unique combinations out of 1.24M loans — almost a 1:1 ratio, 
which defeats the purpose of a dimension table.

**Decision**: Customer attributes (emp_title, emp_length, home_owne

In [ ]:
# ── dim_risk ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_risk AS
    SELECT
        ROW_NUMBER() OVER () AS risk_key,
        sub_grade,
        term,
        mths_since_recent_inq_cat,
        mths_since_recent_bc_cat
    FROM (
        SELECT DISTINCT
            sub_grade, term,
            mths_since_recent_inq_cat, mths_since_recent_bc_cat
        FROM silver
    )
""")

print("dim_risk ✓")
print(con.execute("SELECT COUNT(*) FROM dim_risk").fetchone())

In [ ]:
# ── dim_purpose ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE dim_purpose AS
    SELECT
        ROW_NUMBER() OVER () AS purpose_key,
        purpose
    FROM (SELECT DISTINCT purpose FROM silver)
""")

print("dim_purpose ✓")
print(con.execute("SELECT COUNT(*) FROM dim_purpose").fetchone())
print(con.execute("SELECT * FROM dim_purpose").df())

In [ ]:
# ── fact_loans ──────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE fact_loans AS
    SELECT
        ROW_NUMBER() OVER () AS loan_key,
        dr.risk_key,
        dp.purpose_key,
        dd.date_key,
        -- Customer attributes
        s.emp_title,
        s.emp_length,
        s.home_ownership,
        s.addr_state,
        s.annual_inc,
        s.verification_status,
        s.application_type,
        -- Loan metrics
        s.funded_amnt,
        s.int_rate,
        s.installment,
        s.dti,
        s.revol_bal,
        s.revol_util,
        s.total_pymnt,
        s.out_prncp,
        s.total_bal_ex_mort,
        s.total_rev_hi_lim,
        s.fico_range_low,
        s.credit_history_years,
        -- Credit metrics
        s.delinq_2yrs,
        s.inq_last_6mths,
        s.pub_rec,
        s.total_acc,
        s.acc_now_delinq,
        s.tot_coll_amt,
        s.collections_12_mths_ex_med,
        s.chargeoff_within_12_mths,
        s.delinq_amnt,
        s.mort_acc,
        s.num_accts_ever_120_pd,
        s.num_actv_rev_tl,
        s.num_tl_120dpd_2m,
        s.pct_tl_nvr_dlq,
        s.percent_bc_gt_75,
        s.acc_open_past_24mths,
        -- Target
        s.loan_status,
        s.default_flag
    FROM silver s
    JOIN dim_risk dr 
        ON s.sub_grade = dr.sub_grade 
        AND s.term = dr.term
        AND s.mths_since_recent_inq_cat = dr.mths_since_recent_inq_cat
        AND s.mths_since_recent_bc_cat = dr.mths_since_recent_bc_cat
    JOIN dim_purpose dp ON s.purpose = dp.purpose
    JOIN dim_date dd ON s.issue_d::DATE = dd.issue_d
""")

print("fact_loans ✓")
print(con.execute("SELECT COUNT(*) FROM fact_loans").fetchone())

In [ ]:
# ── Export to Gold ──────────────────────────────────────────────
tables = ['fact_loans', 'dim_risk', 'dim_purpose', 'dim_date']

for table in tables:
    con.execute(f"""
        COPY {table} TO '../data/gold/{table}.csv' (HEADER, DELIMITER ',')
    """)
    print(f"{table} exported ✓")

print("\nGold layer complete!")

In [ ]:
con.close()
print("Connection closed ✓")
